# Ajuste de Curva

### Imports

In [1]:
from numpy.random import randn
from numpy import linspace, zeros, arange, fromiter
from matplotlib.pyplot import figure, rc
from typing import Callable
from numpy import sin, pi

### Ajuste de curva

In [2]:
class CurveAdjust:
  def __init__(self, in1: tuple[int, int], m: int, n: int,
               in2: tuple[int, int], q: int, func: Callable = None,
               nlin: list[tuple[int, Callable]] | Callable = None):

    (n1, n2), self.n = in1, n
    self.in2, self.q = in2, q
    self.nonlinear = nlin

    self.D = D = linspace(n1, n2, m)

    if func is None:
      self.I = I = ((n2 - n1) * randn(m) + n1)
    else:
      self.I = I = fromiter((func(x) for x in D), float, m)

    self.G = G = D.reshape(-1, 1) ** arange(n)
    self.A, self.B = (G.T @ G), (G.T @ I)
    self.fatoraçãoLU(); self.plotgfx()

  def fatoraçãoLU(self):
    L = zeros((self.n, self.n))
    L.flat[::self.n + 1] = 1
    U = self.A

    for i1, i2 in arange(self.n-1).reshape(-1, 1) + (0, 1):
      L[i2:, i1] = U[i2:, i1] / U[i1, i1]
      U[i2:] -= L[i2:, i1, None] * U[i1]

    x, y = zeros(self.n), zeros(self.n)

    for i in arange(self.n):
      y[i] = self.B[i] - sum(L[i] * y)
    for i in arange(self.n)[::-1]:
      x[i] = (y[i] - sum(U[i] * x)) / U[i, i]

    (n1, n2), n, q = self.in2, self.n, self.q
    self.RD = D = linspace(n1, n2, q)

    if isinstance(self.nonlinear, list):
      for i, f in self.nonlinear: x[i] = f(x[i])
    elif isinstance(self.nonlinear, Callable):
      x = self.nonlinear(x)

    self.RI = (D.reshape(-1, 1) ** arange(n)) @ x

  def plotgfx(self):
    rc("font", family = "Consolas", size = 7.5)
    fig = figure(figsize = (12, 5))
    axes = fig.subplots(); fig.dpi = 1000

    axes.scatter(self.D, self.I, label = "Original", aa = 1, s = 1)
    axes.plot(self.RD, self.RI, label = "Ajustado", aa = 1, lw = .5)
    axes.set_title("Comparação do Ajuste de Curva")
    axes.set_xlabel("Domínio das funções")
    axes.set_ylabel("Imagens das funções")
    axes.legend()

    fig.tight_layout(pad = 1)

### Execução do Código

In [3]:
func = lambda x: (sin(pi*abs(x)**15)/(1 + pi*abs(x))) + (1/(1 + 25*x**2))
CurveAdjust((-50, 50), 150, 10, (-50, 50), 1000, func)
func = lambda x: (x**18 - 2**18 - x**8) * (1/(1 + 25*x**2))
CurveAdjust((-50, 50), 150, 10, (-50, 50), 1000, func)
CurveAdjust((-50, 50), 150, 10, (-50, 50), 1000)